In [10]:
import pandas as pd

# === CONFIG ===
src = "data/CollegeBasketballPlayers2009-2021.csv"
out = "data/CollegePlayers_FinalYear_FULL.csv"

# Load
df = pd.read_csv("data/CollegeBasketballPlayers2009-2021.csv")

# Ensure required columns exist
required = ["player_name", "year"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required column(s): {missing}")

# Compute each player's final year
year_num = pd.to_numeric(df["year"], errors="coerce")
final_year = (
    pd.DataFrame({"player_name": df["player_name"], "year_num": year_num})
    .groupby("player_name", as_index=False)["year_num"]
    .max()
    .rename(columns={"year_num": "final_year_num"})
)

# Join to label final-year rows
tmp = df.join(final_year.set_index("player_name"), on="player_name")

# Keep only rows where 'year' equals the player's final year
tmp_year_num = pd.to_numeric(tmp["year"], errors="coerce")
mask_final = tmp_year_num.eq(tmp["final_year_num"])
final_rows = tmp[mask_final].copy()

# Tie-break within the final year: prefer highest GP, then highest Min_per
gp_sort = pd.to_numeric(final_rows.get("GP", 0), errors="coerce").fillna(0)
min_sort = pd.to_numeric(final_rows.get("Min_per", 0), errors="coerce").fillna(0)

final_rows = final_rows.assign(_gp_sort=gp_sort, _min_sort=min_sort)
final_rows = (
    final_rows.sort_values(
        ["player_name", "_gp_sort", "_min_sort"], ascending=[True, False, False]
    )
    .drop_duplicates(subset=["player_name"], keep="first")
    .drop(columns=["final_year_num", "_gp_sort", "_min_sort"], errors="ignore")
)

# Save full dataset (no columns removed)
final_rows.to_csv(out, index=False)

print(f"Saved: {out}")
print(f"Rows kept: {len(final_rows)} (one per player, final season only)")


/var/folders/2d/x3mq62j96lb59syz3fmj1wgr0000gn/T/ipykernel_83732/4196027362.py:8: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/CollegeBasketballPlayers2009-2021.csv")


Saved: data/CollegePlayers_FinalYear_FULL.csv
Rows kept: 25719 (one per player, final season only)
